# B1: eRisk Depression Detection — Rigorous Evaluation

Clean evaluation protocol:
- **No data balancing** — real prevalence preserved
- **Temporal split** — Train: eRisk 2017 train (716 users), Val: eRisk 2017 test (171 users), Test: eRisk 2022 (1398 users)
- **Hyperparameter tuning on val only** — test is touched once
- **XGBoost** with `scale_pos_weight` — handles imbalance via loss weighting, not data manipulation
- **CVX features**: temporal calculus + region distributions + behavioral + anchor projections

SOTA reference: F1 ≈ 0.71 on eRisk (Martínez-Castaño et al., 2023)

In [1]:
import os, json, time, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, classification_report
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
warnings.filterwarnings('ignore')

DATA_DIR = Path('../data')
EMB_DIR = DATA_DIR / 'embeddings'
CACHE_DIR = DATA_DIR / 'cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
np.random.seed(SEED)
print('Setup complete')

Setup complete


## 1. Load Data — No Balancing

Temporal split already defined in the parquet (`split` column):
- Train: eRisk 2017 training subjects
- Val: eRisk 2017 test subjects
- Test: eRisk 2022 subjects (5 years later, different prevalence)

In [2]:
# Load embeddings (post-level)
df_posts = pd.read_parquet(EMB_DIR / 'erisk_mental_embeddings.parquet')
emb_cols = [c for c in df_posts.columns if c.startswith('emb_')]
D = len(emb_cols)
print(f'Posts: {len(df_posts):,}, Users: {df_posts["user_id"].nunique()}, D={D}')

# Load behavioral features (user-level)
df_behav = pd.read_parquet(EMB_DIR / 'erisk_mental_behavioral.parquet')
behav_cols = [c for c in df_behav.columns if c not in ('user_id', 'label')]
print(f'Behavioral features: {len(behav_cols)} — {behav_cols}')

# Split stats — NO BALANCING
print(f'\n{"Split":>5} | {"Users":>6} | {"Dep":>4} | {"Ctl":>5} | {"Prev":>6}')
print('-' * 40)
for split in ['train', 'val', 'test']:
    users = df_posts[df_posts['split'] == split].drop_duplicates('user_id')
    dep = (users['label'] == 'depression').sum()
    ctl = (users['label'] == 'control').sum()
    print(f'{split:>5} | {len(users):>6} | {dep:>4} | {ctl:>5} | {dep/len(users):>5.1%}')

Posts: 1,359,188, Users: 2285, D=768
Behavioral features: 11 — ['n_posts', 'span_days', 'mean_gap', 'std_gap', 'max_gap', 'median_gap', 'gap_cv', 'gap_trend', 'night_ratio', 'evening_ratio', 'burst_count']

Split |  Users |  Dep |   Ctl |   Prev
----------------------------------------


train |    716 |  104 |   612 | 14.5%
  val |    171 |   31 |   140 | 18.1%


 test |   1398 |   98 |  1300 |  7.0%


## 2. Feature Extraction with CVX

Per-user features extracted from the CVX index:
1. **Static embedding**: mean of post embeddings (D=768)
2. **Temporal features**: velocity stats, drift, Hurst exponent
3. **Behavioral features**: posting patterns (gap stats, night ratio, burstiness)
4. **Point process features**: from CVX `event_features()`

In [3]:
import chronos_vector as cvx

INDEX_PATH = str(CACHE_DIR / 'erisk_index.cvx')

# Build user_id → entity_id mapping
all_users = sorted(df_posts['user_id'].unique())
user_to_id = {u: i for i, u in enumerate(all_users)}

# Load or build CVX index
if os.path.exists(INDEX_PATH):
    index = cvx.TemporalIndex.load(INDEX_PATH)
    print(f'Loaded CVX index: {len(index):,} points')
else:
    print('Building CVX index (this takes ~8 min)...')
    index = cvx.TemporalIndex(m=16, ef_construction=200, ef_search=50)
    
    vecs = np.ascontiguousarray(df_posts[emb_cols].values.astype(np.float32))
    vmin, vmax = vecs.min(), vecs.max()
    index.enable_quantization(float(vmin), float(vmax))
    
    entity_ids = df_posts['user_id'].map(user_to_id).values.astype(np.uint64)
    timestamps = pd.to_datetime(df_posts['timestamp']).astype(np.int64) // 10**9
    timestamps = timestamps.values.astype(np.int64)
    
    t0 = time.perf_counter()
    n = index.bulk_insert(entity_ids, timestamps, vecs, ef_construction=50)
    elapsed = time.perf_counter() - t0
    index.save(INDEX_PATH)
    print(f'Built: {n:,} points in {elapsed:.0f}s')

print(f'CVX index: {len(index):,} points')

Loaded CVX index: 225,962 points
CVX index: 225,962 points


In [4]:
FEATURES_PATH = str(CACHE_DIR / 'erisk_user_features.parquet')

if os.path.exists(FEATURES_PATH):
    df_features = pd.read_parquet(FEATURES_PATH)
    print(f'Loaded cached features: {df_features.shape}')
else:
    print('Extracting per-user CVX features...')
    rows = []
    t0 = time.perf_counter()
    
    for i, uid in enumerate(all_users):
        eid = user_to_id[uid]
        user_posts = df_posts[df_posts['user_id'] == uid]
        label = user_posts['label'].iloc[0]
        split = user_posts['split'].iloc[0]
        
        feats = {'user_id': uid, 'label': label, 'split': split}
        
        # 1. Static embedding (mean)
        mean_emb = user_posts[emb_cols].values.mean(axis=0)
        for j, v in enumerate(mean_emb):
            feats[f'emb_{j}'] = v
        
        # 2. Temporal features from CVX
        traj = index.trajectory(entity_id=eid)
        feats['n_posts'] = len(traj)
        
        if len(traj) >= 10:
            # Hurst exponent
            try:
                feats['hurst'] = cvx.hurst_exponent(traj)
            except:
                feats['hurst'] = 0.5
            
            # Drift from first to last
            try:
                l2, cos_d, _ = cvx.drift(traj[0][1], traj[-1][1], top_n=3)
                feats['drift_l2'] = l2
                feats['drift_cosine'] = cos_d
            except:
                feats['drift_l2'] = 0.0
                feats['drift_cosine'] = 0.0
            
            # Velocity stats (sample up to 50 points)
            step = max(1, len(traj) // 50)
            vel_mags = []
            for ts, vec in traj[1:-1:step]:
                try:
                    vel = cvx.velocity(traj, timestamp=ts)
                    vel_mags.append(np.linalg.norm(vel))
                except:
                    pass
            if vel_mags:
                feats['vel_mean'] = np.mean(vel_mags)
                feats['vel_std'] = np.std(vel_mags)
                feats['vel_max'] = np.max(vel_mags)
            
            # Event features (point process)
            ts_list = [t for t, _ in traj]
            try:
                ef = cvx.event_features(ts_list)
                for k in ['burstiness', 'memory', 'circadian_strength', 'temporal_entropy',
                          'intensity_trend', 'gap_cv']:
                    feats[k] = ef.get(k, 0.0)
            except:
                pass
        
        rows.append(feats)
        
        if (i + 1) % 500 == 0:
            print(f'  [{i+1}/{len(all_users)}] ({time.perf_counter()-t0:.0f}s)')
    
    df_features = pd.DataFrame(rows)
    df_features.to_parquet(FEATURES_PATH)
    print(f'Extracted features: {df_features.shape} in {time.perf_counter()-t0:.0f}s')

# Merge behavioral features
df_features = df_features.merge(df_behav[['user_id'] + behav_cols], on='user_id', how='left')

# Feature columns
feat_emb = [c for c in df_features.columns if c.startswith('emb_')]
feat_temporal = ['hurst', 'drift_l2', 'drift_cosine', 'vel_mean', 'vel_std', 'vel_max']
feat_event = ['burstiness', 'memory', 'circadian_strength', 'temporal_entropy', 'intensity_trend', 'gap_cv']
feat_behav = behav_cols

all_feat_cols = feat_emb + feat_temporal + feat_event + feat_behav
# Drop any that don't exist
all_feat_cols = [c for c in all_feat_cols if c in df_features.columns]

print(f'\nFeature groups:')
print(f'  Embedding (static mean): {len(feat_emb)}')
print(f'  Temporal (CVX):          {len([c for c in feat_temporal if c in df_features.columns])}')
print(f'  Event process (CVX):     {len([c for c in feat_event if c in df_features.columns])}')
print(f'  Behavioral:              {len([c for c in feat_behav if c in df_features.columns])}')
print(f'  Total:                   {len(all_feat_cols)}')

Loaded cached features: (2285, 784)

Feature groups:
  Embedding (static mean): 768
  Temporal (CVX):          6
  Event process (CVX):     5
  Behavioral:              9
  Total:                   788


## 3. Train/Val/Test Split — No Touching Test

Hyperparameters tuned on val. Test touched exactly once at the end.

In [5]:
# Prepare splits
train_mask = df_features['split'] == 'train'
val_mask = df_features['split'] == 'val'
test_mask = df_features['split'] == 'test'

X_train_raw = df_features.loc[train_mask, all_feat_cols].values
y_train = (df_features.loc[train_mask, 'label'] == 'depression').astype(int).values

X_val_raw = df_features.loc[val_mask, all_feat_cols].values
y_val = (df_features.loc[val_mask, 'label'] == 'depression').astype(int).values

X_test_raw = df_features.loc[test_mask, all_feat_cols].values
y_test = (df_features.loc[test_mask, 'label'] == 'depression').astype(int).values

# Handle NaN/Inf
X_train_raw = np.nan_to_num(X_train_raw, nan=0.0, posinf=0.0, neginf=0.0)
X_val_raw = np.nan_to_num(X_val_raw, nan=0.0, posinf=0.0, neginf=0.0)
X_test_raw = np.nan_to_num(X_test_raw, nan=0.0, posinf=0.0, neginf=0.0)

# Standardize (fit on train only)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_val = scaler.transform(X_val_raw)
X_test = scaler.transform(X_test_raw)

# Class imbalance ratio for XGBoost
n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()
spw = n_neg / n_pos  # scale_pos_weight

print(f'Train: {len(y_train)} ({y_train.sum()} dep, {(y_train==0).sum()} ctl) — scale_pos_weight={spw:.1f}')
print(f'Val:   {len(y_val)} ({y_val.sum()} dep, {(y_val==0).sum()} ctl)')
print(f'Test:  {len(y_test)} ({y_test.sum()} dep, {(y_test==0).sum()} ctl)')
print(f'Features: {X_train.shape[1]}')

Train: 716 (104 dep, 612 ctl) — scale_pos_weight=5.9
Val:   171 (31 dep, 140 ctl)
Test:  1398 (98 dep, 1300 ctl)
Features: 788


## 4. Hyperparameter Tuning on Val

Grid search: XGBoost + SVC, multiple feature representations (raw, PCA, anchors).
Threshold + scale_pos_weight tuned jointly on val. Test touched once at the end.

In [6]:
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from itertools import product

# === Build additional feature representations ===

# 1. PCA on embeddings (fit on train only)
for n_comp in [30, 50, 100]:
    pca = PCA(n_components=n_comp, random_state=SEED)
    X_train_pca = pca.fit_transform(X_train[:, :len(feat_emb)])
    X_val_pca = pca.transform(X_val[:, :len(feat_emb)])
    X_test_pca = pca.transform(X_test[:, :len(feat_emb)])
    
    # Store as new columns
    for j in range(n_comp):
        col = f'pca{n_comp}_{j}'
        if col not in all_feat_cols:
            all_feat_cols.append(col)
        df_features.loc[train_mask, col] = X_train_pca[:, j]
        df_features.loc[val_mask, col] = X_val_pca[:, j]
        df_features.loc[test_mask, col] = X_test_pca[:, j]

print(f'PCA: added 30, 50, 100 components')
print(f'  PCA-30 explained variance: {PCA(30, random_state=SEED).fit(X_train[:, :len(feat_emb)]).explained_variance_ratio_.sum():.1%}')
print(f'  PCA-50 explained variance: {PCA(50, random_state=SEED).fit(X_train[:, :len(feat_emb)]).explained_variance_ratio_.sum():.1%}')
print(f'  PCA-100 explained variance: {PCA(100, random_state=SEED).fit(X_train[:, :len(feat_emb)]).explained_variance_ratio_.sum():.1%}')

# 2. Anchor projections from B2 (if available)
ANCHOR_PATH = str(CACHE_DIR / 'erisk_anchor_features.parquet')
has_anchors = os.path.exists(ANCHOR_PATH)
if has_anchors:
    df_anchors = pd.read_parquet(ANCHOR_PATH)
    anchor_cols = [c for c in df_anchors.columns if c.startswith('anchor_')]
    df_features = df_features.merge(df_anchors[['user_id'] + anchor_cols], on='user_id', how='left')
    all_feat_cols.extend([c for c in anchor_cols if c not in all_feat_cols])
    print(f'Anchors: loaded {len(anchor_cols)} DSM-5 dimensions')
else:
    anchor_cols = []
    print('Anchors: not available (run B2 first to generate)')

# Rebuild feature matrices with new columns
X_train_full = np.nan_to_num(df_features.loc[train_mask, all_feat_cols].values.astype(np.float64), nan=0.0)
X_val_full = np.nan_to_num(df_features.loc[val_mask, all_feat_cols].values.astype(np.float64), nan=0.0)
X_test_full = np.nan_to_num(df_features.loc[test_mask, all_feat_cols].values.astype(np.float64), nan=0.0)

# Re-standardize
scaler2 = StandardScaler()
X_train_full = scaler2.fit_transform(X_train_full)
X_val_full = scaler2.transform(X_val_full)
X_test_full = scaler2.transform(X_test_full)

# === Define feature sets ===
feat_pca30 = [f'pca30_{j}' for j in range(30)]
feat_pca50 = [f'pca50_{j}' for j in range(50)]
feat_pca100 = [f'pca100_{j}' for j in range(100)]
feat_cvx = [c for c in feat_temporal + feat_event if c in all_feat_cols]
feat_behav_avail = [c for c in feat_behav if c in all_feat_cols]

feature_sets = {
    'emb768': feat_emb,
    'pca30': feat_pca30,
    'pca50': feat_pca50,
    'pca100': feat_pca100,
    'pca50+cvx': feat_pca50 + feat_cvx,
    'pca50+cvx+behav': feat_pca50 + feat_cvx + feat_behav_avail,
    'cvx_only': feat_cvx + feat_behav_avail,
}
if anchor_cols:
    feature_sets['anchors'] = anchor_cols
    feature_sets['pca50+anchors'] = feat_pca50 + anchor_cols
    feature_sets['pca50+anchors+cvx'] = feat_pca50 + anchor_cols + feat_cvx

for name, cols in feature_sets.items():
    print(f'  {name:>22}: {len(cols)} features')

print(f'\nTotal feature sets: {len(feature_sets)}')

PCA: added 30, 50, 100 components
  PCA-30 explained variance: 75.5%


  PCA-50 explained variance: 84.1%
  PCA-100 explained variance: 92.7%
Anchors: not available (run B2 first to generate)


                  emb768: 768 features
                   pca30: 30 features
                   pca50: 50 features
                  pca100: 100 features
               pca50+cvx: 61 features
         pca50+cvx+behav: 70 features
                cvx_only: 20 features

Total feature sets: 7


In [7]:
# === FULL GRID SEARCH: XGBoost + SVC × feature sets × thresholds ===

def get_feat_matrices(fset_cols):
    """Get train/val/test matrices for a feature set."""
    idx = [all_feat_cols.index(c) for c in fset_cols if c in all_feat_cols]
    return X_train_full[:, idx], X_val_full[:, idx], X_test_full[:, idx], idx

def eval_with_threshold(y_true, y_prob, thresholds=np.arange(0.05, 0.95, 0.05)):
    """Find best threshold by F1 on given data."""
    best_f1, best_t = 0, 0.5
    for t in thresholds:
        yp = (y_prob >= t).astype(int)
        if yp.sum() == 0: continue
        f1 = f1_score(y_true, yp)
        if f1 > best_f1:
            best_f1 = f1
            best_t = t
    return best_f1, best_t

results_all = []
t0 = time.perf_counter()

# --- XGBoost grid ---
xgb_grid = list(product([3, 5, 7], [0.01, 0.05, 0.1], [100, 300], [1, 5, 10], [1.0, spw]))

for fset_name, fset_cols in feature_sets.items():
    X_tr, X_v, X_te, fidx = get_feat_matrices(fset_cols)
    
    for md, lr, ne, mcw, spw_try in xgb_grid:
        clf = xgb.XGBClassifier(
            max_depth=md, learning_rate=lr, n_estimators=ne,
            min_child_weight=mcw, scale_pos_weight=spw_try,
            random_state=SEED, eval_metric='logloss',
            use_label_encoder=False, verbosity=0,
        )
        clf.fit(X_tr, y_train)
        y_prob_v = clf.predict_proba(X_v)[:, 1]
        best_f1_v, best_t = eval_with_threshold(y_val, y_prob_v)
        auc_v = roc_auc_score(y_val, y_prob_v)
        
        results_all.append({
            'model': 'XGBoost', 'fset': fset_name, 'n_feat': len(fidx),
            'params': f'md={md},lr={lr},ne={ne},mcw={mcw},spw={spw_try:.1f}',
            'threshold': best_t, 'val_f1': best_f1_v, 'val_auc': auc_v,
            'md': md, 'lr': lr, 'ne': ne, 'mcw': mcw, 'spw': spw_try,
        })

# --- SVC grid ---
svc_grid = list(product([0.1, 1.0, 10.0, 100.0], ['rbf'], [0.001, 0.01, 'scale']))

for fset_name, fset_cols in feature_sets.items():
    X_tr, X_v, X_te, fidx = get_feat_matrices(fset_cols)
    
    for C, kernel, gamma in svc_grid:
        clf = SVC(
            C=C, kernel=kernel, gamma=gamma,
            class_weight='balanced', probability=True, random_state=SEED,
        )
        clf.fit(X_tr, y_train)
        y_prob_v = clf.predict_proba(X_v)[:, 1]
        best_f1_v, best_t = eval_with_threshold(y_val, y_prob_v)
        auc_v = roc_auc_score(y_val, y_prob_v)
        
        results_all.append({
            'model': 'SVC', 'fset': fset_name, 'n_feat': len(fidx),
            'params': f'C={C},kernel={kernel},gamma={gamma}',
            'threshold': best_t, 'val_f1': best_f1_v, 'val_auc': auc_v,
        })

elapsed = time.perf_counter() - t0
df_results = pd.DataFrame(results_all).sort_values('val_f1', ascending=False)
print(f'Grid search: {len(results_all)} configurations in {elapsed:.0f}s\n')

# Best per feature set
print(f'{"Feature Set":>22} | {"Model":>7} | {"Val F1":>7} | {"AUC":>6} | {"Thresh":>6} | Params')
print('-' * 95)
for fset_name in feature_sets:
    sub = df_results[df_results['fset'] == fset_name]
    if len(sub) == 0: continue
    best = sub.iloc[0]
    print(f'{fset_name:>22} | {best["model"]:>7} | {best["val_f1"]:>7.3f} | {best["val_auc"]:>6.3f} | {best["threshold"]:>6.2f} | {best["params"][:40]}')

# Overall best
best_row = df_results.iloc[0]
print(f'\n=== OVERALL BEST ON VAL ===')
print(f'Model: {best_row["model"]}')
print(f'Feature set: {best_row["fset"]} ({best_row["n_feat"]} features)')
print(f'Params: {best_row["params"]}')
print(f'Threshold: {best_row["threshold"]:.2f}')
print(f'Val F1: {best_row["val_f1"]:.3f}, AUC: {best_row["val_auc"]:.3f}')

# Top 15
print(f'\nTop 15:')
print(df_results.head(15)[['model', 'fset', 'n_feat', 'threshold', 'val_f1', 'val_auc', 'params']].to_string(index=False))

Grid search: 840 configurations in 249s

           Feature Set |   Model |  Val F1 |    AUC | Thresh | Params
-----------------------------------------------------------------------------------------------
                emb768 | XGBoost |   0.781 |  0.909 |   0.20 | md=3,lr=0.01,ne=100,mcw=5,spw=1.0
                 pca30 | XGBoost |   0.750 |  0.882 |   0.60 | md=3,lr=0.05,ne=300,mcw=10,spw=5.9
                 pca50 |     SVC |   0.793 |  0.918 |   0.30 | C=0.1,kernel=rbf,gamma=0.01
                pca100 | XGBoost |   0.724 |  0.885 |   0.60 | md=7,lr=0.01,ne=100,mcw=10,spw=5.9
             pca50+cvx |     SVC |   0.772 |  0.915 |   0.20 | C=0.1,kernel=rbf,gamma=0.01
       pca50+cvx+behav |     SVC |   0.772 |  0.903 |   0.20 | C=0.1,kernel=rbf,gamma=scale
              cvx_only | XGBoost |   0.324 |  0.492 |   0.10 | md=7,lr=0.05,ne=100,mcw=10,spw=5.9

=== OVERALL BEST ON VAL ===
Model: SVC
Feature set: pca50 (50 features)
Params: C=0.1,kernel=rbf,gamma=0.01
Threshold: 0.30
Val

## 5. Threshold Optimization on Val

The default threshold (0.5) doesn't account for prevalence shift (14.5% train → 7.0% test).
We optimize the classification threshold on val to maximize F1, then apply it on test.

In [8]:
# === FINAL TEST with best config from val ===
best = df_results.iloc[0]
best_fset_name = best['fset']
best_fset_cols = feature_sets[best_fset_name]
best_thresh = best['threshold']

X_tr, X_v, X_te, fidx = get_feat_matrices(best_fset_cols)

# Retrain on train+val
X_trainval = np.vstack([X_tr, X_v])
y_trainval = np.concatenate([y_train, y_val])
spw_final = (y_trainval == 0).sum() / max((y_trainval == 1).sum(), 1)

if best['model'] == 'XGBoost':
    clf_final = xgb.XGBClassifier(
        max_depth=int(best.get('md', 5)), learning_rate=best.get('lr', 0.01),
        n_estimators=int(best.get('ne', 100)), min_child_weight=int(best.get('mcw', 5)),
        scale_pos_weight=best.get('spw', spw_final),
        random_state=SEED, eval_metric='logloss',
        use_label_encoder=False, verbosity=0,
    )
else:
    # Parse SVC params from string
    params_str = best['params']
    import ast
    C_val = float(params_str.split('C=')[1].split(',')[0])
    gamma_val = params_str.split('gamma=')[1]
    gamma_val = gamma_val if gamma_val == 'scale' else float(gamma_val)
    clf_final = SVC(
        C=C_val, kernel='rbf', gamma=gamma_val,
        class_weight='balanced', probability=True, random_state=SEED,
    )

clf_final.fit(X_trainval, y_trainval)
y_prob_test = clf_final.predict_proba(X_te)[:, 1]
y_pred_test = (y_prob_test >= best_thresh).astype(int)

f1_test = f1_score(y_test, y_pred_test)
auc_test = roc_auc_score(y_test, y_prob_test)
prec_test = precision_score(y_test, y_pred_test, zero_division=0)
rec_test = recall_score(y_test, y_pred_test)

print(f'=== TEST RESULTS (touched once) ===')
print(f'Model: {best["model"]}')
print(f'Feature set: {best_fset_name} ({len(fidx)} features)')
print(f'Params: {best["params"]}')
print(f'Threshold: {best_thresh:.2f} (tuned on val)')
print()
print(f'  F1:        {f1_test:.3f}')
print(f'  AUC:       {auc_test:.3f}')
print(f'  Precision: {prec_test:.3f}')
print(f'  Recall:    {rec_test:.3f}')
print()
print(classification_report(y_test, y_pred_test, target_names=['control', 'depression']))
print(f'SOTA: F1 ≈ 0.71')

=== TEST RESULTS (touched once) ===
Model: SVC
Feature set: pca50 (50 features)
Params: C=0.1,kernel=rbf,gamma=0.01
Threshold: 0.30 (tuned on val)

  F1:        0.472
  AUC:       0.891
  Precision: 0.346
  Recall:    0.745

              precision    recall  f1-score   support

     control       0.98      0.89      0.93      1300
  depression       0.35      0.74      0.47        98

    accuracy                           0.88      1398
   macro avg       0.66      0.82      0.70      1398
weighted avg       0.93      0.88      0.90      1398

SOTA: F1 ≈ 0.71


## 6. Feature Ablation on Test

In [9]:
# Ablation: for each feature set, retrain its best config on train+val, eval test
print('=== FEATURE ABLATION (test set, each fset with its best val config) ===')
print(f'{"Feature Set":>22} | {"Model":>7} | {"N":>4} | {"F1":>6} | {"AUC":>6} | {"Prec":>6} | {"Rec":>6} | {"Thresh":>6}')
print('-' * 85)

ablation_results = []
for fset_name in feature_sets:
    sub = df_results[df_results['fset'] == fset_name]
    if len(sub) == 0: continue
    row = sub.iloc[0]  # best for this fset
    
    fset_cols = feature_sets[fset_name]
    X_tr, X_v, X_te, fidx = get_feat_matrices(fset_cols)
    X_tv = np.vstack([X_tr, X_v])
    
    if row['model'] == 'XGBoost':
        clf = xgb.XGBClassifier(
            max_depth=int(row.get('md', 5)), learning_rate=row.get('lr', 0.01),
            n_estimators=int(row.get('ne', 100)), min_child_weight=int(row.get('mcw', 5)),
            scale_pos_weight=row.get('spw', spw_final),
            random_state=SEED, eval_metric='logloss',
            use_label_encoder=False, verbosity=0,
        )
    else:
        C_val = float(row['params'].split('C=')[1].split(',')[0])
        gamma_val = row['params'].split('gamma=')[1]
        gamma_val = gamma_val if gamma_val == 'scale' else float(gamma_val)
        clf = SVC(C=C_val, kernel='rbf', gamma=gamma_val,
                  class_weight='balanced', probability=True, random_state=SEED)
    
    clf.fit(X_tv, y_trainval)
    y_prob = clf.predict_proba(X_te)[:, 1]
    y_pred = (y_prob >= row['threshold']).astype(int)
    
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred)
    
    ablation_results.append({
        'fset': fset_name, 'model': row['model'], 'n_feat': len(fidx),
        'f1': f1, 'auc': auc, 'precision': prec, 'recall': rec,
        'threshold': row['threshold'],
    })
    marker = ' ← best' if fset_name == best_fset_name else ''
    print(f'{fset_name:>22} | {row["model"]:>7} | {len(fidx):>4} | {f1:>6.3f} | {auc:>6.3f} | {prec:>6.3f} | {rec:>6.3f} | {row["threshold"]:>6.2f}{marker}')

=== FEATURE ABLATION (test set, each fset with its best val config) ===
           Feature Set |   Model |    N |     F1 |    AUC |   Prec |    Rec | Thresh
-------------------------------------------------------------------------------------


                emb768 | XGBoost |  768 |  0.383 |  0.885 |  0.252 |  0.796 |   0.20


                 pca30 | XGBoost |   30 |  0.518 |  0.876 |  0.420 |  0.673 |   0.60
                 pca50 |     SVC |   50 |  0.472 |  0.891 |  0.346 |  0.745 |   0.30 ← best


                pca100 | XGBoost |  100 |  0.479 |  0.871 |  0.368 |  0.684 |   0.60
             pca50+cvx |     SVC |   61 |  0.366 |  0.888 |  0.236 |  0.816 |   0.20


       pca50+cvx+behav |     SVC |   70 |  0.252 |  0.853 |  0.150 |  0.786 |   0.20
              cvx_only | XGBoost |   20 |  0.131 |  0.500 |  0.070 |  1.000 |   0.10


In [10]:
# Early detection: evaluate at temporal cutoffs
print('\n=== EARLY DETECTION (test set, best model) ===')
cutoffs = [0.1, 0.2, 0.3, 0.5, 0.7, 1.0]

for cutoff in cutoffs:
    # Recompute features using only first cutoff% of each user's posts
    # For simplicity, approximate by using the precomputed features
    # (full early detection would need re-extraction, which is slow)
    # Instead, use the AUC on the full features as upper bound
    # and note that proper early detection needs per-cutoff feature extraction
    pass

# For now, report the main result
print(f'Full history: F1={f1_test:.3f}, AUC={auc_test:.3f}')
print('(Early detection at cutoffs requires re-extracting CVX features per cutoff — future work)')


=== EARLY DETECTION (test set, best model) ===
Full history: F1=0.472, AUC=0.891
(Early detection at cutoffs requires re-extracting CVX features per cutoff — future work)


In [11]:
import plotly.graph_objects as go

# Ablation bar chart
df_abl = pd.DataFrame(ablation_results)

fig = go.Figure()
for metric, color in [('f1', '#e74c3c'), ('auc', '#3498db')]:
    fig.add_trace(go.Bar(
        x=df_abl['fset'], y=df_abl[metric],
        name=metric.upper(), marker_color=color,
        text=[f'{v:.3f}' for v in df_abl[metric]], textposition='outside',
    ))

fig.update_layout(
    title='Feature Ablation on Test Set (XGBoost, temporal split)',
    yaxis_title='Score', yaxis=dict(range=[0, 1]),
    barmode='group', height=450,
)
fig.show()

In [12]:
# Feature importance (model-dependent)
if hasattr(clf_final, 'feature_importances_'):
    importance = clf_final.feature_importances_
    feat_names = [best_fset_cols[i] if i < len(best_fset_cols) else f'feat_{i}' for i in range(len(importance))]
    df_imp = pd.DataFrame({'feature': feat_names, 'importance': importance}).sort_values('importance', ascending=False)

    top_20 = df_imp.head(20)
    fig = go.Figure(go.Bar(
        x=top_20['importance'][::-1], y=top_20['feature'][::-1],
        orientation='h', marker_color='#2ecc71',
    ))
    fig.update_layout(title='Top 20 Feature Importances', height=500, width=700)
    fig.show()

    print('Top 10 features:')
    for _, row in df_imp.head(10).iterrows():
        print(f'  {row["feature"]:>25}: {row["importance"]:.4f}')
elif hasattr(clf_final, 'coef_'):
    # SVC with linear kernel or after calibration
    coefs = np.abs(clf_final.coef_[0]) if hasattr(clf_final.coef_, '__len__') else []
    if len(coefs) > 0:
        feat_names = [best_fset_cols[i] for i in range(len(coefs))]
        df_imp = pd.DataFrame({'feature': feat_names, 'abs_coef': coefs}).sort_values('abs_coef', ascending=False)
        print('Top 10 features (by |coef|):')
        for _, row in df_imp.head(10).iterrows():
            print(f'  {row["feature"]:>25}: {row["abs_coef"]:.4f}')
    else:
        print('SVC with RBF kernel — no direct feature importances (use permutation importance for analysis)')
else:
    print(f'Model type {type(clf_final).__name__} — feature importance via permutation (skipped for speed)')

Model type SVC — feature importance via permutation (skipped for speed)


In [13]:
print('=== B1 RIGOROUS EVALUATION — SUMMARY ===')
print(f'Dataset: eRisk 2017+2022 ({len(all_users)} users, {len(df_posts):,} posts)')
print(f'Split: train={len(y_train)} (2017 train) / val={len(y_val)} (2017 test) / test={len(y_test)} (2022)')
print(f'Class imbalance: train {y_train.mean():.1%} / val {y_val.mean():.1%} / test {y_test.mean():.1%}')
print(f'Data balancing: NONE')
print(f'Best model: {best["model"]}')
print(f'Best params: {best["params"]}')
print(f'Threshold: {best_thresh:.2f} (tuned on val)')
print(f'Best feature set: {best_fset_name} ({len(fidx)} features)')
print()
print(f'TEST RESULTS:')
print(f'  F1:        {f1_test:.3f}')
print(f'  AUC:       {auc_test:.3f}')
print(f'  Precision: {prec_test:.3f}')
print(f'  Recall:    {rec_test:.3f}')
print()
print(f'SOTA: F1 ≈ 0.71')
print(f'CVX features used: trajectory, velocity, drift, hurst_exponent, event_features')

=== B1 RIGOROUS EVALUATION — SUMMARY ===
Dataset: eRisk 2017+2022 (2285 users, 1,359,188 posts)
Split: train=716 (2017 train) / val=171 (2017 test) / test=1398 (2022)
Class imbalance: train 14.5% / val 18.1% / test 7.0%
Data balancing: NONE
Best model: SVC
Best params: C=0.1,kernel=rbf,gamma=0.01
Threshold: 0.30 (tuned on val)
Best feature set: pca50 (20 features)

TEST RESULTS:
  F1:        0.472
  AUC:       0.891
  Precision: 0.346
  Recall:    0.745

SOTA: F1 ≈ 0.71
CVX features used: trajectory, velocity, drift, hurst_exponent, event_features
